# Energy usage prediction: the best model + a neural network (`candidate_nn.csv`)

This notebook tests whether a **neural network** improves our best public-leaderboard submission (`candidate_v8.csv`, public RMSE 3.27026). It rebuilds that model exactly (the code in Parts B and C is copied unchanged from `best_model.ipynb`), adds a neural network, checks the blend with cross-validation, and writes **`candidate_nn.csv`**.

## The idea in one picture

```
                      ┌──────────────────────────────────────────────┐
  train.csv ─────────►│ Part B: "v4" model                           │──► v4 ──────────┐ 25%
                      └──────────────────────────────────────────────┘                 ▼
                      ┌──────────────────────────────────────────────┐            best model ──┐ (1 - w)
  train.csv ─────────►│ Part C: LightGBM-imputed model (seed 527)    │──► C ───────────▲ 75%   │
  test.csv (inputs) ─►│   imputers fill blanks ────────────────────┐ │                         ▼
                      └────────────────────────────────────────────┼─┘                  final prediction
                      ┌────────────────────────────────────────────▼─┐                         ▲
                      │ Part D: neural network (NEW)                 │──► neural net ──────────┘ w
                      │   5 networks predict the change from         │
                      │   previous_usage, using C's filled-in values │
                      └──────────────────────────────────────────────┘
```

**Why a small weight `w` and not a neural network on its own?** In a quick test, a neural network alone scored worse than our per-building Ridge (5-fold RMSE 3.24 vs 3.18), but its errors are slightly different, so a 20–35% blend helped by about 0.02. Part E measures this properly on the full pipeline.

**Why scikit-learn and not PyTorch?** The platform environment (`datathon-platform-env`) has scikit-learn, LightGBM and XGBoost but no PyTorch or TensorFlow. `MLPRegressor` is a standard fully connected neural network, which is the usual deep-learning choice for a small table like ours.

## Part A: settings, data and shared helpers

In [1]:
import time
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import sklearn
import lightgbm
from sklearn.experimental import enable_iterative_imputer  # must be imported before IterativeImputer
from sklearn.impute import IterativeImputer
from sklearn.ensemble import ExtraTreesRegressor, HistGradientBoostingRegressor
from sklearn.linear_model import Ridge, RidgeCV
from sklearn.model_selection import KFold
from sklearn.neural_network import MLPRegressor
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from threadpoolctl import threadpool_limits

warnings.filterwarnings('ignore')

# ---------------- settings you might want to change ----------------
DATA_FOLDER = Path('.')                         # folder that contains train.csv and test.csv
OUTPUT_FILE = DATA_FOLDER / 'candidate_nn_w05.csv'
RETRAIN_V4 = False        # False: reuse candidate_v4.csv, exactly as best_model.ipynb does
RANDOM_SEEDS = [527]                            # LightGBM-imputer seed of the 3.27 submission
WEIGHT_ON_LIGHTGBM_MODEL = 0.75                 # the existing best blend: 75% LightGBM-imputed model, 25% v4

NN_NETWORKS = 5                # neural networks averaged together (each starts from different random weights)
NN_HIDDEN_LAYERS = (256, 128)  # neurons in each hidden layer
NN_L2_PENALTY = 1e-2           # weight decay; larger = smoother network
RUN_CROSS_VALIDATION = True    # about 6 minutes; measures the neural network and picks its weight
NN_WEIGHT = 0.05               # 5%: best on the plain and test-like-blank CV scores and on the public scores (see Results);
                               # 'auto' = weight picked by the weighted CV score (0.15 -> candidate_nn.csv, public 3.27148)
NN_WEIGHT_IF_NO_CV = 0.2       # used when NN_WEIGHT is 'auto' but cross-validation is switched off
CV_FOLDS = 5
CPU_THREADS = 4

print('scikit-learn', sklearn.__version__, '| lightgbm', lightgbm.__version__, '| pandas', pd.__version__)

scikit-learn 1.5.2 | lightgbm 4.5.0 | pandas 2.2.3


In [2]:
train = pd.read_csv(DATA_FOLDER / 'train.csv')
test = pd.read_csv(DATA_FOLDER / 'test.csv')
test_inputs = test.drop(columns=['energy_usage'], errors='ignore')   # test.csv has no answers anyway

NUMERIC_COLUMNS = ['temperature', 'humidity', 'occupancy', 'previous_usage']   # the only columns with blanks
BUILDING_IDS = ['ADM_A', 'BUS_A', 'BUS_B', 'ENG_A', 'ENG_B', 'LEC_A', 'LIB_A', 'RES_A', 'RES_B', 'SCI_A', 'SCI_B', 'SPT_A']
BUILDING_TYPES = ['Administration', 'Business', 'Engineering', 'LectureHall', 'Library', 'Residential', 'Science', 'Sports']
DAY_NAMES = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']
DAY_NUMBER = {day: number for number, day in enumerate(DAY_NAMES)}
BUILDING_NUMBER = {building: number for number, building in enumerate(BUILDING_IDS)}

print(f'train: {len(train):,} rows | test: {len(test):,} rows')
print('blank values in test:', test_inputs[NUMERIC_COLUMNS].isna().sum().to_dict())

train: 8,000 rows | test: 3,000 rows
blank values in test: {'temperature': 211, 'humidity': 183, 'occupancy': 206, 'previous_usage': 203}


### Unusual rows and test-mix weights

The test set has about **five times as many unusual "event" rows** as the training set. Each row is compared with the typical value for its building, hour and weekday/weekend, and the first matching event is recorded:

| Event number | Event | Rule |
|---|---|---|
| 1 | heat | temperature of 33.5 or more |
| 2 | rain | humidity of 93 or more |
| 3 | occupancy surge | 100 or more people above typical |
| 4 | occupancy drop | 60 or more people below typical |
| 5 | previous-usage spike | more than 1.6 times typical |
| 6 | previous-usage drop | less than half of typical |

`test_mix_weights` gives each training row a weight, so the weighted training data has the same building × event mix as the test inputs. The counts are smoothed toward the overall event mix, so rare combinations don't get extreme weights, and every weight is clipped to between 0.2 and 10.

In [3]:
def is_weekend(rows):
    return rows['day_of_week'].isin(['Saturday', 'Sunday'])


def event_type(rows, reference_rows):
    """0 = normal row, 1-6 = the first event (see table above) that applies to the row."""
    group_keys = ['building_id', 'hour', 'weekend']
    reference = reference_rows.assign(weekend=is_weekend(reference_rows))
    typical = reference.groupby(group_keys)[['occupancy', 'previous_usage']].median().add_suffix('_typical')
    joined = rows.assign(weekend=is_weekend(rows)).join(typical, on=group_keys)

    occupancy_gap = joined['occupancy'] - joined['occupancy_typical']
    previous_ratio = joined['previous_usage'] / joined['previous_usage_typical']
    event_flags = np.column_stack([
        joined['temperature'] >= 33.5,     # 1 heat
        joined['humidity'] >= 93,          # 2 rain
        occupancy_gap >= 100,              # 3 occupancy surge
        occupancy_gap <= -60,              # 4 occupancy drop
        previous_ratio > 1.6,              # 5 previous-usage spike
        previous_ratio < 0.5,              # 6 previous-usage drop
    ])                                     # blank values count as "no event"
    first_event = np.argmax(event_flags, axis=1) + 1
    return pd.Series(np.where(event_flags.any(axis=1), first_event, 0), index=rows.index)


def test_mix_weights(source_rows, target_rows, smoothing=15):
    """Weight for each source row so the source's building x event mix matches the target's."""
    source_events = event_type(source_rows, source_rows)
    target_events = event_type(target_rows, source_rows)       # "typical" values always come from the source rows
    all_events = range(7)

    source_counts = pd.crosstab(source_rows['building_id'], source_events).reindex(index=BUILDING_IDS, columns=all_events, fill_value=0)
    target_counts = pd.crosstab(target_rows['building_id'], target_events).reindex(index=BUILDING_IDS, columns=all_events, fill_value=0)

    # share of each event within each building, smoothed toward the overall event mix
    source_overall_mix = (np.bincount(source_events, minlength=7) + 1) / (len(source_events) + 7)
    target_overall_mix = (np.bincount(target_events, minlength=7) + 1) / (len(target_events) + 7)
    source_event_share = (source_counts + smoothing * source_overall_mix).div(source_counts.sum(axis=1) + smoothing, axis=0)
    target_event_share = (target_counts + smoothing * target_overall_mix).div(target_counts.sum(axis=1) + smoothing, axis=0)
    event_ratio = target_event_share / source_event_share

    # how much more (or less) common each building is in the target
    building_ratio = (target_counts.sum(axis=1) / len(target_rows)) / (source_counts.sum(axis=1) / len(source_rows))

    weights = [event_ratio.loc[building, event] * building_ratio[building]
               for building, event in zip(source_rows['building_id'], source_events)]
    return np.clip(np.array(weights), 0.2, 10.0)


train_events = event_type(train, train)
test_events = event_type(test_inputs, train)
print(f'unusual event rows: train {(train_events > 0).mean():.1%} | test {(test_events > 0).mean():.1%}')

unusual event rows: train 5.1% | test 26.1%


## Parts B and C: the existing best model (unchanged)

The next three cells are copied **unchanged** from `best_model.ipynb`; see that notebook for the full explanation.

* **B1 / B2: the "v4" model** = 25% v3 notebook model (MissForest + Ridge / gradient boosting) + 75% one model per pattern of missing inputs.
* **C: the LightGBM-imputed model.** LightGBM models fill in blank values, then 80% Ridge + 20% gradient boosting.

Best model = 25% v4 + 75% C. Part F checks that this notebook reproduces `candidate_v8.csv` exactly before blending in the neural network.

In [4]:
def missforest_inputs(rows):
    """The four numeric columns (may contain blanks) plus never-blank time and building columns."""
    inputs = rows[NUMERIC_COLUMNS].copy()
    inputs['hour_sin'] = np.sin(2 * np.pi * rows['hour'] / 24)
    inputs['hour_cos'] = np.cos(2 * np.pi * rows['hour'] / 24)
    inputs['month_sin'] = np.sin(2 * np.pi * rows['month'] / 12)
    inputs['month_cos'] = np.cos(2 * np.pi * rows['month'] / 12)
    inputs['is_weekend'] = is_weekend(rows).astype(float)
    building_onehot = pd.get_dummies(pd.Categorical(rows['building_id'], categories=BUILDING_IDS), prefix='bid').astype(float)
    building_onehot.index = rows.index
    return pd.concat([inputs, building_onehot], axis=1)


def fit_missforest(training_rows):
    imputer = IterativeImputer(
        estimator=ExtraTreesRegressor(n_estimators=100, min_samples_leaf=3, max_features=0.6, n_jobs=CPU_THREADS, random_state=0),
        max_iter=8, random_state=0)
    return imputer.fit(missforest_inputs(training_rows))


def fill_blanks_with_missforest(imputer, rows):
    filled = rows.copy()
    filled[NUMERIC_COLUMNS] = imputer.transform(missforest_inputs(rows))[:, :len(NUMERIC_COLUMNS)]
    return filled


def typical_values(filled_rows):
    """Median occupancy and previous usage for each building, hour and weekday/weekend."""
    return (filled_rows.assign(is_weekend=is_weekend(filled_rows))
            .groupby(['building_id', 'hour', 'is_weekend'])[['occupancy', 'previous_usage']].median())


def ridge_features(rows, typical):
    """Features for the Ridge model (rows must have no blanks)."""
    features = rows[NUMERIC_COLUMNS].copy()
    features['hour_sin'] = np.sin(2 * np.pi * rows['hour'] / 24)
    features['hour_cos'] = np.cos(2 * np.pi * rows['hour'] / 24)
    features['month_sin'] = np.sin(2 * np.pi * rows['month'] / 12)
    features['month_cos'] = np.cos(2 * np.pi * rows['month'] / 12)
    features['is_weekend'] = is_weekend(rows).astype(float)
    features['occupancy_x_weekend'] = features['occupancy'] * features['is_weekend']

    building_onehot = pd.get_dummies(pd.Categorical(rows['building_id'], categories=BUILDING_IDS), prefix='bid').astype(float)
    type_onehot = pd.get_dummies(pd.Categorical(rows['building_type'], categories=BUILDING_TYPES), prefix='bt').astype(float)
    building_onehot.index = type_onehot.index = rows.index

    interactions = {}
    # each building gets its own slope for the main drivers
    for column in ['occupancy', 'previous_usage', 'temperature', 'hour_sin', 'hour_cos']:
        for building in building_onehot.columns:
            interactions[f'{column}_x_{building}'] = features[column] * building_onehot[building]
    # weekend effect per building type
    for column in ['is_weekend', 'occupancy_x_weekend']:
        for building_type in type_onehot.columns:
            interactions[f'{column}_x_{building_type}'] = features[column] * type_onehot[building_type]

    # how unusual the row is compared with its building's typical values
    lookup_keys = pd.DataFrame({'building_id': rows['building_id'], 'hour': rows['hour'], 'is_weekend': features['is_weekend'] > 0})
    typical_for_row = lookup_keys.join(typical, on=['building_id', 'hour', 'is_weekend']).fillna(typical.median())
    events = pd.DataFrame({
        'occupancy_gap': features['occupancy'] - typical_for_row['occupancy'],
        'previous_ratio': features['previous_usage'] / typical_for_row['previous_usage'],
        'previous_gap': features['previous_usage'] - typical_for_row['previous_usage'],
        'heat': (features['temperature'] >= 33.5).astype(float),
        'rain': (features['humidity'] >= 93).astype(float),
    }, index=rows.index)
    for column in events.columns:
        for building_type in type_onehot.columns:
            interactions[f'{column}_x_{building_type}'] = events[column] * type_onehot[building_type]

    # occupancy effect by part of the day, per building type
    part_of_day = pd.cut(rows['hour'], [-1, 5, 8, 11, 14, 17, 20, 23], labels=False)
    part_onehot = pd.get_dummies(pd.Categorical(part_of_day, categories=range(7)), prefix='dp').astype(float)
    part_onehot.index = rows.index
    for part in part_onehot.columns:
        for building_type in type_onehot.columns:
            interactions[f'occupancy_{part}_x_{building_type}'] = features['occupancy'] * part_onehot[part] * type_onehot[building_type]

    return pd.concat([features, type_onehot, building_onehot, events, pd.DataFrame(interactions, index=rows.index)], axis=1)


def tree_features(rows):
    """Features for the gradient-boosting model (raw values; building columns treated as categories)."""
    features = rows[['hour', 'month'] + NUMERIC_COLUMNS].copy()
    features['day'] = rows['day_of_week'].map(DAY_NUMBER)
    features['is_weekend'] = (features['day'] >= 5).astype(int)
    features['building_type'] = pd.Categorical(rows['building_type'], categories=BUILDING_TYPES)
    features['building_id'] = pd.Categorical(rows['building_id'], categories=BUILDING_IDS)
    return features


def new_gradient_boosting():
    return HistGradientBoostingRegressor(categorical_features='from_dtype', max_iter=1500, learning_rate=0.03,
                                         max_leaf_nodes=15, min_samples_leaf=30, l2_regularization=1.0,
                                         early_stopping=True, n_iter_no_change=50, random_state=42)


class V3NotebookModel:
    def fit(self, training_rows):
        self.imputer = fit_missforest(training_rows)                    # learns from training rows only
        filled = fill_blanks_with_missforest(self.imputer, training_rows)
        target = filled['energy_usage']
        self.typical = typical_values(filled)
        self.ridge = make_pipeline(StandardScaler(), RidgeCV(alphas=np.logspace(-3, 3, 13)))
        self.ridge.fit(ridge_features(filled, self.typical), target)
        self.boosting = new_gradient_boosting()
        self.boosting.fit(tree_features(filled), target - filled['previous_usage'])     # predicts the change
        return self

    def predict(self, rows):
        filled = fill_blanks_with_missforest(self.imputer, rows)
        ridge_prediction = self.ridge.predict(ridge_features(filled, self.typical))
        boosting_prediction = self.boosting.predict(tree_features(filled)) + filled['previous_usage'].to_numpy()
        return 0.8 * ridge_prediction + 0.2 * boosting_prediction

In [5]:
def pattern_model_inputs(rows, available_columns, n_daily_curves):
    """Available numeric columns, a weekend flag, daily sine/cosine curves and a yearly sine/cosine curve."""
    hour = rows['hour'].to_numpy(float)
    columns = {name: rows[name].to_numpy(float) for name in available_columns}
    columns['weekend'] = is_weekend(rows).to_numpy(float)
    for k in range(1, n_daily_curves + 1):                     # k = 1, 2, 3: once, twice, three times a day
        columns[f'hour_sin_{k}'] = np.sin(2 * np.pi * k * hour / 24)
        columns[f'hour_cos_{k}'] = np.cos(2 * np.pi * k * hour / 24)
    columns['month_sin'] = np.sin(2 * np.pi * rows['month'].to_numpy() / 12)
    columns['month_cos'] = np.cos(2 * np.pi * rows['month'].to_numpy() / 12)
    return np.column_stack(list(columns.values()))


def blank_pattern_number(rows):
    """0-15: bit k is set when NUMERIC_COLUMNS[k] is blank (0 = nothing blank)."""
    blanks = rows[NUMERIC_COLUMNS].isna().to_numpy().astype(int)
    return (blanks * 2 ** np.arange(len(NUMERIC_COLUMNS))).sum(axis=1)


class MissingPatternModels:
    def __init__(self, n_daily_curves=3, ridge_alpha=1.0, correction_weight=0.65,
                 correction_trees=250, correction_min_leaf=25, correction_max_leaves=15, test_mix_power=0.5):
        self.n_daily_curves = n_daily_curves
        self.ridge_alpha = ridge_alpha
        self.correction_weight = correction_weight
        self.correction_trees = correction_trees
        self.correction_min_leaf = correction_min_leaf
        self.correction_max_leaves = correction_max_leaves
        self.test_mix_power = test_mix_power

    def _ridge_inputs(self, rows, available_columns, scaler):
        scaled = scaler.transform(pattern_model_inputs(rows, available_columns, self.n_daily_curves))
        building_onehot = (rows['building_id'].to_numpy()[:, None] == np.array(BUILDING_IDS)[None, :]).astype(float)
        per_building = (scaled[:, :, None] * building_onehot[:, None, :]).reshape(len(scaled), -1)   # every input x every building
        return np.column_stack([scaled, building_onehot, per_building])

    @staticmethod
    def _correction_inputs(rows, available_columns):
        numeric = rows[['hour', 'month'] + available_columns].to_numpy(float)
        return np.column_stack([numeric, rows['day_of_week'].map(DAY_NUMBER), rows['building_id'].map(BUILDING_NUMBER)])

    def fit(self, training_rows, test_rows):
        weights = test_mix_weights(training_rows, test_rows) ** self.test_mix_power
        weights = pd.Series(weights / weights.mean(), index=training_rows.index)
        self.models = {}
        for pattern in range(2 ** len(NUMERIC_COLUMNS)):
            available = [column for bit, column in enumerate(NUMERIC_COLUMNS) if not pattern & (1 << bit)]
            usable_rows = training_rows.dropna(subset=available)          # rows that have every available column
            scaler = StandardScaler().fit(pattern_model_inputs(usable_rows, available, self.n_daily_curves))
            ridge_inputs = self._ridge_inputs(usable_rows, available, scaler)
            row_weights = weights.loc[usable_rows.index].to_numpy()

            ridge = Ridge(alpha=self.ridge_alpha).fit(ridge_inputs, usable_rows['energy_usage'], sample_weight=row_weights)
            correction_inputs = self._correction_inputs(usable_rows, available)
            correction = HistGradientBoostingRegressor(
                max_iter=self.correction_trees, max_leaf_nodes=self.correction_max_leaves, learning_rate=0.05,
                min_samples_leaf=self.correction_min_leaf, l2_regularization=8.0, early_stopping=False, random_state=42,
                categorical_features=[False] * (correction_inputs.shape[1] - 1) + [True])   # last column = building
            correction.fit(correction_inputs, usable_rows['energy_usage'].to_numpy() - ridge.predict(ridge_inputs),
                           sample_weight=row_weights)
            self.models[pattern] = (available, scaler, ridge, correction)
        return self

    def predict(self, rows):
        patterns = blank_pattern_number(rows)
        prediction = np.empty(len(rows))
        for pattern in np.unique(patterns):
            positions = np.flatnonzero(patterns == pattern)
            pattern_rows = rows.iloc[positions]
            available, scaler, ridge, correction = self.models[pattern]
            prediction[positions] = (ridge.predict(self._ridge_inputs(pattern_rows, available, scaler))
                                     + self.correction_weight * correction.predict(self._correction_inputs(pattern_rows, available)))
        return prediction

In [6]:
def v4_predictions(training_rows, test_rows):
    v3_model = V3NotebookModel().fit(training_rows)
    pattern_models = MissingPatternModels().fit(training_rows, test_rows)
    return 0.25 * v3_model.predict(test_rows) + 0.75 * pattern_models.predict(test_rows)

In [7]:
IMPUTER_CATEGORIES = {'building_id': BUILDING_IDS, 'building_type': BUILDING_TYPES, 'day_of_week': DAY_NAMES}


def imputer_inputs(rows):
    """Everything the imputers may look at (the column being predicted is dropped later)."""
    inputs = rows[['building_id', 'building_type', 'hour', 'day_of_week', 'month'] + NUMERIC_COLUMNS].copy()
    inputs['weekend'] = is_weekend(rows).astype(int)
    inputs['hour_sin'] = np.sin(2 * np.pi * rows['hour'] / 24)
    inputs['hour_cos'] = np.cos(2 * np.pi * rows['hour'] / 24)
    inputs['month_sin'] = np.sin(2 * np.pi * rows['month'] / 12)
    inputs['month_cos'] = np.cos(2 * np.pi * rows['month'] / 12)
    for column, categories in IMPUTER_CATEGORIES.items():          # text columns -> LightGBM categories
        inputs[column] = pd.Categorical(inputs[column], categories=categories)
    return inputs


def new_lightgbm_imputer(seed):
    return lightgbm.LGBMRegressor(n_estimators=2200, learning_rate=0.025, max_depth=5, num_leaves=32,
                                  min_child_samples=30, reg_lambda=15.0, subsample=0.8, subsample_freq=1,
                                  colsample_bytree=0.8, cat_smooth=20, verbose=-1, random_state=seed, n_jobs=CPU_THREADS)


class LightGBMImputedModel:
    def __init__(self, seed, practice_copies=3, test_mix_power=1.5, boosting_share=0.2):
        self.seed = seed
        self.practice_copies = practice_copies
        self.test_mix_power = test_mix_power
        self.boosting_share = boosting_share

    # ---- C1: the imputers ----
    def fit_imputers(self, training_rows, test_rows):
        all_inputs = pd.concat([training_rows.drop(columns='energy_usage'), test_rows], ignore_index=True)
        random_generator = np.random.default_rng(self.seed)
        weights = test_mix_weights(all_inputs, test_rows) ** self.test_mix_power
        weights = weights / weights.mean()
        test_blank_patterns = test_rows[NUMERIC_COLUMNS].isna().to_numpy()

        self.imputers = {}
        for column in NUMERIC_COLUMNS:
            observed = all_inputs.loc[all_inputs[column].notna()].copy()   # rows where this column is known
            practice_sets = [observed]
            for _ in range(self.practice_copies):
                copy = observed.copy()
                blanks = test_blank_patterns[random_generator.integers(0, len(test_rows), len(copy))]
                blanks[:, NUMERIC_COLUMNS.index(column)] = False            # keep the column being learned
                copy[NUMERIC_COLUMNS] = copy[NUMERIC_COLUMNS].mask(blanks)
                practice_sets.append(copy)
            practice_data = pd.concat(practice_sets, ignore_index=True)
            practice_weights = np.tile(weights[observed.index.to_numpy()], self.practice_copies + 1)

            imputer = new_lightgbm_imputer(self.seed)
            imputer.fit(imputer_inputs(practice_data).drop(columns=column), practice_data[column], sample_weight=practice_weights)
            self.imputers[column] = imputer
        return self

    def expected_value(self, rows, column):
        """What the imputer thinks `column` should be, given the rest of the row."""
        return self.imputers[column].predict(imputer_inputs(rows).drop(columns=column))

    def fill_blanks(self, rows):
        filled = rows.copy()
        for column in NUMERIC_COLUMNS:
            blank = rows[column].isna()
            if blank.any():
                filled.loc[blank, column] = self.expected_value(rows.loc[blank], column)
        return filled

    # ---- C2: the energy model ----
    def model_features(self, filled_rows, original_rows):
        features = ridge_features(filled_rows, self.typical).copy()
        expected_previous = self.expected_value(original_rows, 'previous_usage')
        expected_occupancy = self.expected_value(original_rows, 'occupancy')
        features['expected_previous'] = expected_previous
        features['expected_occupancy'] = expected_occupancy
        for building_type in BUILDING_TYPES:
            this_type = (filled_rows['building_type'] == building_type).to_numpy(float)
            features[f'expected_previous_{building_type}'] = expected_previous * this_type
            features[f'expected_occupancy_{building_type}'] = expected_occupancy * this_type
        return features

    def fit(self, training_rows, test_rows):
        self.fit_imputers(training_rows, test_rows)
        filled = self.fill_blanks(training_rows)
        self.typical = typical_values(filled)
        self.ridge = make_pipeline(StandardScaler(), RidgeCV(alphas=np.logspace(-3, 3, 13)))
        self.ridge.fit(self.model_features(filled, training_rows), filled['energy_usage'])
        self.boosting = new_gradient_boosting()
        self.boosting.fit(tree_features(filled), filled['energy_usage'] - filled['previous_usage'])
        return self

    def predict(self, rows):
        filled = self.fill_blanks(rows)
        ridge_prediction = self.ridge.predict(self.model_features(filled, rows))
        boosting_prediction = self.boosting.predict(tree_features(filled)) + filled['previous_usage'].to_numpy()
        return (1 - self.boosting_share) * ridge_prediction + self.boosting_share * boosting_prediction

## Part D: the neural network

**Inputs** (scaled to mean 0, standard deviation 1). Blanks are first filled by Part C's LightGBM imputers, so the network sees the same filled-in values as the Ridge model:

* the four numeric columns: temperature, humidity, occupancy, previous usage;
* the imputers' **expected** previous usage and occupancy for the row;
* the event features: how far occupancy and previous usage are from their typical value for the building, hour and weekday/weekend, plus heat and rain flags;
* time: a weekend flag, three daily sine/cosine curves and a yearly sine/cosine curve;
* building and building type (one-hot).

**Target:** the **change** from `previous_usage` (energy usage − previous usage), like the gradient-boosting models. In the quick test this beat predicting energy usage directly (3.24 vs 3.29 on its own).

**Network:** two hidden layers of 256 and 128 neurons (ReLU), Adam optimiser, weight decay 0.01, batches of 128 rows. Training stops when the score on a held-back 10% of the training rows hasn't improved for 25 passes. **Five networks** with different random starting weights are averaged, which steadies the predictions.

In [8]:
class NeuralNetModel:
    """Neural networks that predict the change from previous_usage, using Part C's filled-in values."""

    def __init__(self, n_networks=NN_NETWORKS, hidden_layers=NN_HIDDEN_LAYERS, l2_penalty=NN_L2_PENALTY):
        self.n_networks = n_networks
        self.hidden_layers = hidden_layers
        self.l2_penalty = l2_penalty

    @staticmethod
    def network_inputs(lightgbm_model, filled_rows, original_rows):
        hour = filled_rows['hour'].to_numpy(float)
        weekend = is_weekend(filled_rows).to_numpy()
        columns = {name: filled_rows[name].to_numpy(float) for name in NUMERIC_COLUMNS}
        columns['expected_previous'] = lightgbm_model.expected_value(original_rows, 'previous_usage')
        columns['expected_occupancy'] = lightgbm_model.expected_value(original_rows, 'occupancy')

        # how unusual the row is compared with its building's typical values (same lookup as ridge_features)
        typical = lightgbm_model.typical
        lookup_keys = pd.DataFrame({'building_id': filled_rows['building_id'], 'hour': filled_rows['hour'], 'is_weekend': weekend})
        typical_for_row = lookup_keys.join(typical, on=['building_id', 'hour', 'is_weekend']).fillna(typical.median())
        columns['occupancy_gap'] = columns['occupancy'] - typical_for_row['occupancy'].to_numpy()
        columns['previous_gap'] = columns['previous_usage'] - typical_for_row['previous_usage'].to_numpy()
        columns['previous_ratio'] = columns['previous_usage'] / typical_for_row['previous_usage'].to_numpy()
        columns['heat'] = (columns['temperature'] >= 33.5).astype(float)
        columns['rain'] = (columns['humidity'] >= 93).astype(float)

        columns['weekend'] = weekend.astype(float)
        for k in range(1, 4):                                       # once, twice, three times a day
            columns[f'hour_sin_{k}'] = np.sin(2 * np.pi * k * hour / 24)
            columns[f'hour_cos_{k}'] = np.cos(2 * np.pi * k * hour / 24)
        columns['month_sin'] = np.sin(2 * np.pi * filled_rows['month'].to_numpy() / 12)
        columns['month_cos'] = np.cos(2 * np.pi * filled_rows['month'].to_numpy() / 12)

        building_onehot = (filled_rows['building_id'].to_numpy()[:, None] == np.array(BUILDING_IDS)[None, :]).astype(float)
        type_onehot = (filled_rows['building_type'].to_numpy()[:, None] == np.array(BUILDING_TYPES)[None, :]).astype(float)
        return np.column_stack(list(columns.values()) + [building_onehot, type_onehot])

    def fit(self, lightgbm_model, training_rows):
        filled = lightgbm_model.fill_blanks(training_rows)
        inputs = self.network_inputs(lightgbm_model, filled, training_rows)
        self.scaler = StandardScaler().fit(inputs)
        change = (filled['energy_usage'] - filled['previous_usage']).to_numpy()
        self.change_mean, self.change_std = change.mean(), change.std()

        self.networks = []
        for seed in range(self.n_networks):
            network = MLPRegressor(hidden_layer_sizes=self.hidden_layers, alpha=self.l2_penalty, batch_size=128,
                                   learning_rate_init=1e-3, max_iter=500, early_stopping=True, validation_fraction=0.1,
                                   n_iter_no_change=25, random_state=seed)
            network.fit(self.scaler.transform(inputs), (change - self.change_mean) / self.change_std)
            self.networks.append(network)
        return self

    def predict(self, lightgbm_model, rows):
        filled = lightgbm_model.fill_blanks(rows)
        inputs = self.scaler.transform(self.network_inputs(lightgbm_model, filled, rows))
        change = np.mean([network.predict(inputs) for network in self.networks], axis=0)
        return change * self.change_std + self.change_mean + filled['previous_usage'].to_numpy()

## Part E: cross-validation (does the neural network help, and how much weight should it get?)

Five folds. In each fold, every model is trained on 80% of `train.csv` and predicts the other 20%, which plays the role of the test set (it is what the imputers and the test-mix weights see). Three scores are reported:

| Score | What it measures |
|---|---|
| **as is** | plain RMSE on the held-out rows |
| **test-like blanks** | the held-out rows get extra blanks copied from random test rows, because test rows have about five times as many blanks |
| **test-like (weighted)** | test-like blanks, and each row weighted by `test_mix_weights`, so the building × event mix matches the test set. This is the closest to the leaderboard, so `NN_WEIGHT = 'auto'` uses it |

The neural network uses its own Part C imputers in each fold, exactly as in the final fit.

In [9]:
def rmse(prediction, truth, weights=None):
    return float(np.sqrt(np.average((prediction - truth) ** 2, weights=weights)))


def with_test_like_blanks(rows, seed):
    """Copy of rows with extra blanks copied from random test rows (existing blanks stay blank)."""
    random_generator = np.random.default_rng(seed)
    test_blank_patterns = test_inputs[NUMERIC_COLUMNS].isna().to_numpy()
    blanks = test_blank_patterns[random_generator.integers(0, len(test_inputs), len(rows))]
    masked = rows.copy()
    masked[NUMERIC_COLUMNS] = masked[NUMERIC_COLUMNS].mask(blanks)
    return masked


MODEL_NAMES = ['v4', 'lightgbm', 'neural_net']
SCORE_KINDS = ['as_is', 'test_blanks']
truth = train['energy_usage'].to_numpy()
row_weights = test_mix_weights(train, test_inputs)          # how test-like each training row is
cv_predictions = None

if RUN_CROSS_VALIDATION:
    cv_predictions = {f'{name}_{kind}': np.zeros(len(train)) for name in MODEL_NAMES for kind in SCORE_KINDS}
    folds = KFold(CV_FOLDS, shuffle=True, random_state=0)
    with threadpool_limits(limits=CPU_THREADS):
        for fold, (fit_positions, check_positions) in enumerate(folds.split(train)):
            started = time.time()
            fit_rows = train.iloc[fit_positions]
            check_rows = {'as_is': train.iloc[check_positions].drop(columns='energy_usage')}
            check_rows['test_blanks'] = with_test_like_blanks(check_rows['as_is'], seed=fold)

            v3_model = V3NotebookModel().fit(fit_rows)
            pattern_models = MissingPatternModels().fit(fit_rows, check_rows['test_blanks'])
            lightgbm_model = LightGBMImputedModel(seed=RANDOM_SEEDS[0]).fit(fit_rows, check_rows['test_blanks'])
            neural_net = NeuralNetModel().fit(lightgbm_model, fit_rows)

            for kind, rows in check_rows.items():
                cv_predictions[f'v4_{kind}'][check_positions] = 0.25 * v3_model.predict(rows) + 0.75 * pattern_models.predict(rows)
                cv_predictions[f'lightgbm_{kind}'][check_positions] = lightgbm_model.predict(rows)
                cv_predictions[f'neural_net_{kind}'][check_positions] = neural_net.predict(lightgbm_model, rows)

            fold_truth = truth[check_positions]
            scores = ' | '.join(f"{name} {rmse(cv_predictions[f'{name}_test_blanks'][check_positions], fold_truth):.3f}"
                                for name in MODEL_NAMES)
            print(f'fold {fold}: test-like blanks RMSE: {scores}   ({time.time() - started:.0f}s)')

    np.savez(DATA_FOLDER / 'experiments' / 'neural_net_cv.npz', truth=truth, row_weights=row_weights, **cv_predictions)
    print('saved fold predictions to experiments/neural_net_cv.npz')

fold 0: test-like blanks RMSE: v4 3.221 | lightgbm 3.241 | neural_net 3.373   (38s)
fold 1: test-like blanks RMSE: v4 3.239 | lightgbm 3.285 | neural_net 3.448   (36s)
fold 2: test-like blanks RMSE: v4 3.220 | lightgbm 3.238 | neural_net 3.380   (36s)
fold 3: test-like blanks RMSE: v4 3.442 | lightgbm 3.392 | neural_net 3.499   (40s)
fold 4: test-like blanks RMSE: v4 3.330 | lightgbm 3.307 | neural_net 3.386   (39s)
saved fold predictions to experiments/neural_net_cv.npz


In [10]:
def three_scores(prediction_for_kind):
    return {'as is': rmse(prediction_for_kind('as_is'), truth),
            'test-like blanks': rmse(prediction_for_kind('test_blanks'), truth),
            'test-like (weighted)': rmse(prediction_for_kind('test_blanks'), truth, row_weights)}


def best_model_cv(kind):
    return ((1 - WEIGHT_ON_LIGHTGBM_MODEL) * cv_predictions[f'v4_{kind}']
            + WEIGHT_ON_LIGHTGBM_MODEL * cv_predictions[f'lightgbm_{kind}'])


if cv_predictions is not None:
    model_table = pd.DataFrame({
        'v4 (Part B)': three_scores(lambda kind: cv_predictions[f'v4_{kind}']),
        'LightGBM-imputed (Part C)': three_scores(lambda kind: cv_predictions[f'lightgbm_{kind}']),
        'neural network alone (Part D)': three_scores(lambda kind: cv_predictions[f'neural_net_{kind}']),
        'best model (25% v4 + 75% C)': three_scores(best_model_cv),
    }).T
    print('Cross-validation RMSE of each model (lower is better)')
    display(model_table.round(4))

    blend_table = pd.DataFrame({
        weight: three_scores(lambda kind, w=weight: (1 - w) * best_model_cv(kind) + w * cv_predictions[f'neural_net_{kind}'])
        for weight in np.round(np.arange(0, 0.55, 0.05), 2)}).T
    blend_table.index.name = 'neural-net weight'
    print('Cross-validation RMSE of (1 - w) x best model + w x neural network')
    display(blend_table.round(4))

    error_correlation = np.corrcoef(best_model_cv('test_blanks') - truth, cv_predictions['neural_net_test_blanks'] - truth)[0, 1]
    print(f'correlation between the best model\'s errors and the neural network\'s errors: {error_correlation:.3f}')

if NN_WEIGHT == 'auto':
    nn_weight = float(blend_table['test-like (weighted)'].idxmin()) if cv_predictions is not None else NN_WEIGHT_IF_NO_CV
else:
    nn_weight = float(NN_WEIGHT)
print(f'neural-net weight used for {OUTPUT_FILE.name}: {nn_weight:.2f}')

Cross-validation RMSE of each model (lower is better)


,as is,test-like blanks,test-like (weighted)
v4 (Part B),3.0804,3.2913,3.6869
LightGBM-imputed (Part C),3.0844,3.2931,3.6497
neural network alone (Part D),3.2225,3.4176,3.7600
best model (25% v4 + 75% C),3.0742,3.2791,3.6394


Cross-validation RMSE of (1 - w) x best model + w x neural network


,as is,test-like blanks,test-like (weighted)
neural-net weight,,,
0.00,3.0742,3.2791,3.6394
0.05,3.0741,3.2789,3.6373
0.10,3.0748,3.2796,3.6361
0.15,3.0763,3.2809,3.6357
0.20,3.0786,3.2831,3.6362
0.25,3.0817,3.2860,3.6375
0.30,3.0856,3.2896,3.6397
0.35,3.0904,3.2940,3.6428
0.40,3.0959,3.2992,3.6468


correlation between the best model's errors and the neural network's errors: 0.957
neural-net weight used for candidate_nn_w05.csv: 0.05


## Part F: final fit and submission file

1. **v4**: loaded from `candidate_v4.csv` (or retrained), as in `best_model.ipynb`.
2. **Part C**: fitted on all of `train.csv`, with the imputers also seeing the `test.csv` inputs (seed 527).
3. **Neural network**: fitted on all of `train.csv`, using that Part C model's imputers.
4. **Check**: 25% v4 + 75% C must match `candidate_v8.csv` (the 3.27026 submission).
5. **Blend**: `(1 - w) × best model + w × neural network`, saved to `candidate_nn.csv`.

In [11]:
saved_v4_file = DATA_FOLDER / 'candidate_v4.csv'
if not RETRAIN_V4 and saved_v4_file.exists():
    saved_v4 = pd.read_csv(saved_v4_file)
    assert saved_v4['id'].equals(test['id']), 'candidate_v4.csv rows are not in test.csv order'
    v4_prediction = saved_v4['prediction'].to_numpy()
    print('v4: loaded candidate_v4.csv (the exact predictions that scored 3.37)')
else:
    with threadpool_limits(limits=CPU_THREADS):
        v4_prediction = v4_predictions(train, test_inputs)
    print('v4: retrained from scratch')

lightgbm_predictions, neural_net_predictions = [], []
with threadpool_limits(limits=CPU_THREADS):
    for seed in RANDOM_SEEDS:
        lightgbm_model = LightGBMImputedModel(seed=seed).fit(train, test_inputs)
        lightgbm_predictions.append(lightgbm_model.predict(test_inputs))
        neural_net = NeuralNetModel().fit(lightgbm_model, train)
        neural_net_predictions.append(neural_net.predict(lightgbm_model, test_inputs))
        print(f'seed {seed} done')

lightgbm_model_prediction = np.mean(lightgbm_predictions, axis=0)
neural_net_prediction = np.mean(neural_net_predictions, axis=0)
best_model_prediction = (1 - WEIGHT_ON_LIGHTGBM_MODEL) * v4_prediction + WEIGHT_ON_LIGHTGBM_MODEL * lightgbm_model_prediction

# check: the best-model part must reproduce the 3.27026 submission
submitted_file = DATA_FOLDER / 'candidate_v8.csv'
if submitted_file.exists():
    submitted = pd.read_csv(submitted_file)
    largest_difference = np.abs(submitted['prediction'].to_numpy() - best_model_prediction).max()
    print(f'best-model part vs candidate_v8.csv (3.27026): same row order {submitted["id"].equals(test["id"])}, '
          f'largest difference {largest_difference:.2e}')

gap = neural_net_prediction - best_model_prediction
print(f'neural network vs best model on test: typical difference {np.sqrt(np.mean(gap ** 2)):.3f}, '
      f'largest {np.abs(gap).max():.2f}, correlation {np.corrcoef(neural_net_prediction, best_model_prediction)[0, 1]:.4f}')

v4: loaded candidate_v4.csv (the exact predictions that scored 3.37)
seed 527 done
best-model part vs candidate_v8.csv (3.27026): same row order True, largest difference 4.26e-14
neural network vs best model on test: typical difference 1.089, largest 6.13, correlation 0.9986


In [12]:
final_prediction = (1 - nn_weight) * best_model_prediction + nn_weight * neural_net_prediction
assert len(final_prediction) == len(test) and np.isfinite(final_prediction).all()
submission = pd.DataFrame({'id': test['id'], 'prediction': final_prediction})
submission.to_csv(OUTPUT_FILE, index=False)

print(f'wrote {OUTPUT_FILE.name}: {len(submission):,} rows | columns {list(submission.columns)} | '
      f'blanks: {submission["prediction"].isna().sum()} | neural-net weight {nn_weight:.2f}')
print(f'typical change from candidate_v8.csv: {np.sqrt(np.mean((final_prediction - best_model_prediction) ** 2)):.3f}')
submission.head()

wrote candidate_nn_w05.csv: 3,000 rows | columns ['id', 'prediction'] | blanks: 0 | neural-net weight 0.05
typical change from candidate_v8.csv: 0.054


,id,prediction
0,EC010541,71.706870
1,EC009755,66.233314
2,EC008886,56.100256
3,EC009317,64.349953
4,EC008958,21.205882


## Results

**Cross-validation RMSE** (5 folds, lower is better; see Part E for what each score means)

| Model | as is | test-like blanks | test-like (weighted) |
|---|---|---|---|
| v4 (Part B) | 3.080 | 3.291 | 3.687 |
| LightGBM-imputed (Part C) | 3.084 | 3.293 | 3.650 |
| neural network alone (Part D) | 3.223 | 3.418 | 3.760 |
| best model (25% v4 + 75% C) = `candidate_v8.csv` recipe | 3.074 | 3.279 | 3.639 |
| best model + 5% neural network = `candidate_nn_w05.csv` | 3.074 | 3.279 | 3.637 |
| best model + 15% neural network = `candidate_nn.csv` | 3.076 | 3.281 | 3.636 |

**Public leaderboard**

| File | Neural-net weight | Public RMSE |
|---|---|---|
| `candidate_v8.csv` | 0% | 3.27026 |
| `candidate_nn.csv` | 15% | 3.27148 |
| `candidate_nn_w05.csv` | 5% | _not submitted yet_ (predicted ≈ 3.2698) |

**What the two public scores tell us.** RMSE² is a parabola in the weight `w`: `MSE(w) = MSE(0) + 2·w·c + w²·D`, where `D` = 1.186 is the mean of (neural net − v8)² on the test rows, which we can compute. The two public scores then give `c` = −0.062, so the best public weight is `−c / D` ≈ **0.05**, worth about **0.0005** (3.27026 → about 3.2698). Cross-validation agrees: 5% is also best on the plain and test-like-blank scores. 15% overshot.

**Other adjustments tried (on the saved fold predictions in `experiments/neural_net_cv.npz`, with nested cross-validation: weights learned on 4 folds and scored on the 5th):**

* **Different neural-net weight for normal rows and unusual "event" rows.** The network helps more on occupancy and previous-usage events (best weight 0.35–0.55 within those rows), but in nested cross-validation the gain shrinks to ≤ 0.0035 on the weighted score and is slightly negative on the plain score. The public scores also show the network helps less on public rows than cross-validation suggests (`c` = −0.06 public vs −0.14 expected). **Not used.**
* **Bias correction per event type.** The best model seemed to under-predict occupancy surges by about 0.9, but that came from surge rows whose `occupancy` was blanked out, where nothing can reveal the surge. With event flags computed from the inputs, the bias is within noise, and correcting it doesn't help. **Not used.**

**Bottom line:** the neural network is worth about 0.0005 at best. The remaining error is mostly noise (rows with every input present are already at about 3.07) and information lost to blanks, which no blend weight can recover.